# Leakage Randomized Benchmarking

## Background: leakage in quantum processors

Superconducting qubits (transmons) are not true two-level systems — they are weakly anharmonic oscillators with a ladder of energy levels $|0\rangle, |1\rangle, |2\rangle, \ldots$ The computational qubit is encoded in the lowest two levels, but microwave control pulses can inadvertently excite population into the $|2\rangle$ state (or higher). This process is called **leakage**.

Leakage is particularly insidious because:

- **It is invisible to standard readout.** Most dispersive readout schemes only distinguish $|0\rangle$ from $|1\rangle$; a qubit in $|2\rangle$ may be misclassified as $|1\rangle$.
- **It is coherent and persistent.** Unlike depolarizing noise, leaked population does not naturally decay back to the computational subspace on gate timescales. It accumulates over a circuit.
- **It defeats standard randomized benchmarking.** Standard RB assumes errors remain within the computational subspace, so it conflates leakage with depolarization in the extracted error rate.

To model leakage, we represent each qubit as a **qutrit** and, after each gate, apply a noise channel obtained by evolving a Lindbladian generator that combines three processes:

| Channel | Quax generator | Effect |
|---------|---------------|--------|
| **Leakage** | `qx.lindbladians.leakage(γ)` | Population from $\|1\rangle$ leaks to $\|2\rangle$ at rate $\gamma$ |
| **Seepage** | `qx.lindbladians.seepage(γ)` | Population from $\|2\rangle$ returns to $\|1\rangle$ at rate $\gamma$ |
| **Depolarizing** | `qx.lindbladians.depolarizing(p, (2,))` | Standard depolarizing within the $\|0\rangle$–$\|1\rangle$ subspace |

## Leakage randomized benchmarking (LRB)

LRB, introduced by [Wood & Gambetta (2018)](https://arxiv.org/abs/1704.03081), extends standard RB by fitting **two** decay curves simultaneously:

1. **Computational subspace population** $p_\mathrm{comp}(m)$: the probability that the qubit is still in $\{|0\rangle, |1\rangle\}$ after $m$ gates. This decays as $A \cdot \lambda_1^m + B$ and yields the leakage rate $L_1 = 1 - \lambda_1$.
2. **Ground state survival** $p_0(m)$: the probability of measuring $|0\rangle$, which decays as $A' \cdot \lambda_2^m + B'$ and encodes the combined effect of leakage and depolarizing errors.

By fitting both curves, LRB independently extracts the leakage rate $L_1$, seepage rate $L_2$, and average gate infidelity $E$.

## This notebook

We construct random Clifford sequences, promote them into a qutrit Hilbert space, apply a Lindbladian-based noise model (depolarizing + leakage + seepage), and fit the LRB decay models to independently extract these rates.

In [ ]:
%load_ext autoreload
%autoreload 2

## Imports

We use `quax` for quantum object manipulation (unitaries, density matrices, superoperators), `jax` for automatic differentiation and JIT compilation, `optax` for gradient-based curve fitting, and `plotly` for visualization.

In [ ]:
from functools import reduce
from operator import mul

import jax
import jax.numpy as jnp

import quax as qx

try:
    import optax
except ImportError:
    print("optax not found, optimization will not work. Install optax to enable optimization features.")

try:
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
except ImportError:
    print("plotly not found, plotting will not work. Install plotly to enable plotting features.")

## Parameters

Define the simulation parameters. The **depths** list controls the RB sequence lengths, and **num_randomizations** sets how many random Clifford sequences are sampled at each depth. The three error rates — depolarizing, leakage, and seepage — define the noise model applied after each gate. We will later verify that the fitting procedure recovers these known rates from the simulated data.

In [ ]:
seed = 3847
"""The random seed for reproducibility."""

depths = [1, 3, 7, 13, 27, 53, 81, 121, 161, 201, 401, 801]
"""The depth of the RB circuits."""

num_randomizations = 30
"""The number of random RB circuits to generate for each depth."""

depolarizing_error = 0.03
"""The depolarizing error rate to apply to each gate in the RB circuits."""

leakage_rate = 0.003
"""The leakage rate to apply to each gate in the RB circuits."""

seepage_rate = 0.005
"""The seepage rate to apply to each gate in the RB circuits.""";

## Build the circuits

Construct the RB circuits by randomly sampling elements of the single-qubit Clifford group. For each depth, we draw `depth - 1` random Cliffords, compute their cumulative product, and append the inverse so the ideal sequence compiles to the identity. Shorter sequences are padded with identity gates to a uniform length (`max_depth`) so that all depths can be stacked into a single array and processed in parallel with `vmap`.

In [ ]:
key = jax.random.key(seed)

num_cliffords = qx.ensembles.CLIFFORDS_1Q.ensemble_size[0]
max_depth = max(depths)
depths_arr = jnp.array(depths)


def identity_unitary(dims: tuple[int, ...], ensemble_size: tuple[int, ...] = ()) -> qx.Unitary:
    """Create an identity Unitary with the given qudit dimensions and optional ensemble shape."""
    d = reduce(mul, dims)
    eye = jnp.eye(d, dtype=complex)
    return qx.Unitary.from_matrix(jnp.broadcast_to(eye, ensemble_size + (d, d)), (dims, dims))


def compose_sequence(unitaries: qx.Unitary) -> qx.Unitary:
    """Compose a sequence of unitaries via scan. Input: (depth, ..., 2, 2)."""
    init = identity_unitary(dims=(2,), ensemble_size=unitaries.ensemble_size[1:])
    result, _ = jax.lax.scan(lambda acc, u: (u @ acc, None), init, unitaries)
    return result


# Generate, invert, and pad each depth's Clifford sequence to max_depth
all_cliffords_data = []
for depth in depths:
    subkey, key = jax.random.split(key)
    rand_ints = jax.random.randint(subkey, shape=(depth - 1, num_randomizations), minval=0, maxval=num_cliffords)
    random_cliffords = qx.Unitary(data=qx.ensembles.CLIFFORDS_1Q.data[rand_ints], num_qubits=1)

    # Compose all Cliffords and append the inverse (adjoint)
    inversion = compose_sequence(random_cliffords).h
    full_data = jnp.concatenate([random_cliffords.data, inversion.data[jnp.newaxis]], axis=0)

    # Pad shorter sequences to max_depth with identity gates
    pad_length = max_depth - depth
    if pad_length > 0:
        pad = identity_unitary(dims=(2,), ensemble_size=(pad_length, num_randomizations))
        full_data = jnp.concatenate([full_data, pad.data], axis=0)
    all_cliffords_data.append(full_data)

# Stack: (num_depths, max_depth, num_randomizations, 2, 2)
all_cliffords = qx.Unitary(data=jnp.stack(all_cliffords_data), num_qubits=1)

# Mask: True where real gates exist (positions 0..depth-1 for each depth)
gate_mask = jnp.arange(max_depth)[None, :] < depths_arr[:, None]

## Check the final state of each random sequence

As a sanity check, we apply each noiseless Clifford sequence to the $|0\rangle$ state and verify that the probability of measuring $|0\rangle$ is 1.0 at every depth. This confirms that the inversion gate correctly compiles each sequence back to the identity.

In [ ]:
def apply_sequence(unitaries: qx.Unitary) -> qx.StateVector:
    """Apply a sequence of unitaries to |0⟩."""
    ensemble_size = unitaries.ensemble_size[1:]
    initial_state = qx.zero_state_vector(1, ensemble_size)
    final_state, _ = jax.lax.scan(lambda state, u: (u @ state, None), initial_state, unitaries)
    return final_state


all_final_states = jax.vmap(apply_sequence)(all_cliffords)
all_zero_probs = qx.probabilities(all_final_states)[..., 0]

for i, depth in enumerate(depths):
    print(f"Probability of measuring '0' at depth {depth}: {all_zero_probs[i].mean():.4f}")

## Promote to qutrit space

To model leakage we need a third energy level $|2\rangle$. We use `qx.promote` to embed the $2 \times 2$ Clifford unitaries into $3 \times 3$ qutrit unitaries, where the promoted gate acts as identity on the $|2\rangle$ subspace.

In [ ]:
# Promote all cliffords to qutrit space
CLIFFORDS_QUTRIT = qx.promote(all_cliffords, (3,))

## Build the noise model and simulate

We construct the per-gate noise as a single **Lindbladian generator** — the canonical GKSL description of open-system noise — combining **depolarizing** noise (restricted to the qubit subspace), **leakage** ($|1\rangle \to |2\rangle$), and **seepage** ($|2\rangle \to |1\rangle$). Adding Lindbladian generators is the physically correct way to combine independent noise processes; the qubit-subspace depolarizing generator is auto-promoted into the qutrit space when added to the leakage/seepage generators. We then `qx.evolve` the combined generator once for unit time to obtain a single CPTP `SuperOp` applied after every gate. The noisy gate superoperators are applied sequentially to an initial $|0\rangle\langle 0|$ density matrix using `jax.lax.scan`, and all depths and randomizations are processed in parallel via `jax.vmap` and `jax.jit`.

In [ ]:
def compute_final_matrix(random_sequences: qx.SuperOp) -> qx.DensityMatrix:
    """
    Scan-reduce a sequence of SuperOps applied to |0><0|.

    :param random_sequences: (depth, ..., 3, 3, 3, 3) ensemble of SuperOps.
    :return: (...) final density matrix.
    """
    ensemble_size = random_sequences.ensemble_size[1:]
    initial_state = qx.zero_state_matrix(dims=(3,), ensemble_size=ensemble_size)
    final_state, _ = jax.lax.scan(lambda state, channel: (channel @ state, None), initial_state, random_sequences)
    return final_state


def compute_all_probabilities(random_sequences: qx.SuperOp):
    """Compute all qutrit probabilities from a noisy channel sequence."""
    final_dm = compute_final_matrix(random_sequences)
    return qx.probabilities(final_dm)  # (..., 3) -> [P(|0>), P(|1>), P(|2>)]


def probability_ensemble(depolarizing_error, leakage_rate, seepage_rate, qutrit_cliffords, gate_mask):
    """
    Build the noise channel, apply it to the promoted cliffords, and compute all probabilities.

    :param depolarizing_error: Depolarizing error rate (scalar).
    :param leakage_rate: Leakage rate (scalar).
    :param seepage_rate: Seepage rate (scalar).
    :param qutrit_cliffords: Promoted Clifford unitaries (num_depths, max_depth, num_randomizations, 3, 3).
    :param gate_mask: Boolean mask (num_depths, max_depth) — True for real gates.
    :return: all_probs of shape (num_depths, num_randomizations, 3).
    """
    # Build the per-gate noise as a single Lindbladian generator, then evolve it once for unit
    # time into a CPTP SuperOp.  Adding generators is the physically correct way to combine
    # independent noise processes (unlike composing/summing the channels themselves): here
    # depolarizing on the qubit subspace (auto-promoted to the qutrit space when added to the
    # qutrit generators), leakage (|1> -> |2>), and seepage (|2> -> |1>).
    noise_generator = (
        qx.lindbladians.depolarizing(jnp.asarray(depolarizing_error), dims=(2,))
        + qx.lindbladians.leakage(jnp.asarray(leakage_rate))
        + qx.lindbladians.seepage(jnp.asarray(seepage_rate))
    )
    noise = qx.evolve(noise_generator, 1.0)

    # Compose noise with all gate unitaries at once (@ handles Unitary → SuperOp conversion)
    all_noisy = noise @ qutrit_cliffords

    # Replace padded positions with identity SuperOp
    identity_super = qx.unitary_to_superop(identity_unitary(dims=(3,)))
    mask_nd = gate_mask[..., None, None, None, None, None]
    all_channels_data = jnp.where(mask_nd, all_noisy.data, identity_super.data)
    all_channels = qx.SuperOp(data=all_channels_data, num_qubits=1)

    # vmap compute_all_probabilities over the depths dimension
    return jax.vmap(compute_all_probabilities)(all_channels)


# JIT-compile and run — returns (num_depths, num_randomizations, 3)
all_probs = jax.jit(probability_ensemble)(depolarizing_error, leakage_rate, seepage_rate, CLIFFORDS_QUTRIT, gate_mask)

# LRB observables (Wood & Gambetta Eqs. 9, 15)
# p_𝟙₁(m): computational subspace population = P(|0⟩) + P(|1⟩) = 1 - P(|2⟩)
# p₀(m): ground state survival probability = P(|0⟩)
avg_p_comp = jnp.mean(1 - all_probs[..., 2], axis=1)
avg_p0 = jnp.mean(all_probs[..., 0], axis=1)

for i, depth in enumerate(depths):
    print(f"Depth {depth:3d}: p_𝟙₁ = {avg_p_comp[i]:.4f}, p₀ = {avg_p0[i]:.4f}")

## Plot the LRB observables vs depth

Following Wood & Gambetta, we plot the two key LRB observables: the computational-subspace population $p_{\mathbb{1}_1}(m) = \text{Tr}[\mathbb{1}_1 \, \rho(m)]$ and the ground-state survival probability $p_0(m)$. Both should exhibit exponential decay as a function of sequence depth $m$.

In [ ]:
def plot_lrb(depths, y_left, y_right, fit_left=None, fit_right=None):
    """Plot p_𝟙₁(m) and p₀(m) side-by-side with optional fit curves (m_fine, y_fine, label)."""
    fig = make_subplots(rows=1, cols=2, subplot_titles=("p_𝟙₁(m) vs depth", "p₀(m) vs depth"))
    marker_kw = {"size": 10, "line": {"width": 2, "color": "DarkSlateGrey"}}
    panels = [
        (y_left, "p_𝟙₁", "#00b5ad", "p_𝟙₁(m)", [0, 1.05], fit_left),
        (y_right, "p₀", "#ef476f", "p₀(m)", [0, 1.05], fit_right),
    ]
    for col, (y, name, color, ytitle, yrange, fit) in enumerate(panels, 1):
        mode = "markers" if fit is not None else "lines+markers"
        fig.add_trace(
            go.Scatter(
                x=depths,
                y=y.tolist(),
                mode=mode,
                name=name,
                marker={"color": color, **marker_kw},
                line={"color": color},
            ),
            row=1,
            col=col,
        )
        if fit is not None:
            m_fine, y_fine, label = fit
            fig.add_trace(
                go.Scatter(x=m_fine.tolist(), y=y_fine.tolist(), mode="lines", name=label, line={"color": color}),
                row=1,
                col=col,
            )
        fig.update_xaxes(title_text="Sequence depth m", row=1, col=col)
        fig.update_yaxes(title_text=ytitle, range=yrange, row=1, col=col)
    fig.update_layout(height=450, width=1000, template="ggplot2", showlegend=fit_left is not None)
    fig.show()


plot_lrb(depths, avg_p_comp, avg_p0)

## Fit the LRB decay models

We follow the LRB protocol of [Wood & Gambetta](https://arxiv.org/abs/1704.03081) to extract the leakage rate $L_1$, seepage rate $L_2$, and average gate infidelity $E$.

**Step 1 — Leakage model (Eq. 9):** Fit the computational-subspace population to

$$p_{\mathbb{1}_1}(m) = A + B\,\lambda_1^m$$

and extract the leakage and seepage rates (Eqs. 10–11):

$$L_1 = (1 - A)(1 - \lambda_1), \qquad L_2 = A\,(1 - \lambda_1)$$

**Step 2 — Fidelity model (Eq. 15):** Using $\lambda_1$ from Step 1, fit the ground-state survival probability to the three-parameter model

$$p_0(m) = A_0 + B_0\,\lambda_1^m + C_0\,\lambda_2^m$$

and extract the average gate fidelity (Eq. 16):

$$\bar{F} = \frac{1}{d_1}\bigl[(d_1 - 1)\,\lambda_2 + 1 - L_1\bigr]$$

In [ ]:
p_comp = jnp.array(avg_p_comp)
p0 = jnp.array(avg_p0)
m_jnp = jnp.array(depths, dtype=float)


# Leakage decay model: p_𝟙₁(m) = A + B·λ₁ᵐ  (WG Eq. 9)
def leakage_decay(params, m):
    A, B, lam1 = params
    return A + B * lam1**m


def fit_lbfgs(model_fn, m, y, init_params, num_steps=200):
    """Fit model_fn(params, m) to data y using L-BFGS."""
    params = jnp.array(init_params)
    solver = optax.lbfgs()
    opt_state = solver.init(params)

    def loss_fn(p):
        return jnp.mean((model_fn(p, m) - y) ** 2)

    value_and_grad_fn = jax.value_and_grad(loss_fn)

    @jax.jit
    def step(params, opt_state):
        value, grad = value_and_grad_fn(params)
        updates, opt_state_new = solver.update(
            grad,
            opt_state,
            params,
            value=value,
            grad=grad,
            value_fn=loss_fn,
        )
        return optax.apply_updates(params, updates), opt_state_new

    for _ in range(num_steps):
        params, opt_state = step(params, opt_state)
    return params


# --- Step 1: Fit p_𝟙₁(m) = A + B·λ₁ᵐ  (Eq. 9) ---
popt_leak = fit_lbfgs(leakage_decay, m_jnp, p_comp, [0.6, 0.4, 0.99])
A, B, lam1 = popt_leak

L1 = (1 - A) * (1 - lam1)  # Leakage rate  (Eq. 10)
L2 = A * (1 - lam1)  # Seepage rate  (Eq. 11)


# --- Step 2: Fit p₀(m) = A₀ + B₀·λ₁ᵐ + C₀·λ₂ᵐ  (Eq. 15, λ₁ fixed) ---
def fidelity_decay(params, m):
    A0, B0, C0, lam2 = params
    return A0 + B0 * lam1**m + C0 * lam2**m


popt_fid = fit_lbfgs(fidelity_decay, m_jnp, p0, [0.3, 0.1, 0.5, 0.95])
A0, B0, C0, lam2 = popt_fid

d1 = 2  # computational subspace dimension (qubit)
F_avg = ((d1 - 1) * lam2 + 1 - L1) / d1  # Average gate fidelity (Eq. 16)
E = 1 - F_avg  # Average gate infidelity

### Extracted parameters

From the two fits we extract the three LRB parameters of Wood & Gambetta:

- **Leakage rate** $L_1 = (1-A)(1-\lambda_1)$
- **Seepage rate** $L_2 = A(1-\lambda_1)$
- **Average gate infidelity** $E = 1 - \bar{F}$, where $\bar{F} = [(d_1-1)\lambda_2 + 1 - L_1]/d_1$

#### What should $L_1$ and $L_2$ come out to?

It is tempting to compare these against the nominal `leakage_rate` and `seepage_rate`, but that is **not** what LRB measures. Two corrections are needed.

**1. Subspace averaging (the factor of $d_1$).** Wood & Gambetta define the leakage rate as the leakage probability averaged over the *whole* computational subspace,

$$L_1 = \mathrm{Tr}\bigl[\mathbb{1}_2 \, \Lambda(\mathbb{1}_1 / d_1)\bigr],$$

i.e. starting from the maximally mixed state $(|0\rangle\langle 0| + |1\rangle\langle 1|)/2$. But `qx.lindbladians.leakage(γ)` has jump operator $\sqrt{\gamma}\,|2\rangle\langle 1|$ — only $|1\rangle$ leaks, while $|0\rangle$ does not. Averaging over both computational levels therefore **halves** the rate: $L_1 \approx \gamma_L / d_1$, not $\gamma_L$.

Seepage escapes this correction entirely: the leakage subspace is one-dimensional ($d_2 = 1$), so there is nothing to average over and $L_2 \approx \gamma_S$.

**2. Rates are not probabilities.** We build the channel by evolving the generator for unit time, so the $|1\rangle \to |2\rangle$ population transfer is not $\gamma_L$ but $(\gamma_L/\Gamma)(1 - e^{-\Gamma})$ with $\Gamma = \gamma_L + \gamma_S$ — leakage and seepage compete during the evolution. This is a sub-percent effect at these rates, but it is easy to include.

Together these give the values LRB should actually recover:

$$L_1 = \frac{1}{d_1}\,\frac{\gamma_L}{\Gamma}\bigl(1 - e^{-\Gamma}\bigr), \qquad L_2 = \frac{\gamma_S}{\Gamma}\bigl(1 - e^{-\Gamma}\bigr), \qquad \Gamma = \gamma_L + \gamma_S.$$

We compare against these below. The gate infidelity $E$ has no such simple closed form — it captures the combined effect of the depolarizing, leakage, and seepage channels.

In [ ]:
# Expected LRB rates for this noise model.  These are *not* the nominal rates: L₁ is averaged over
# the computational subspace (only |1> leaks, so it picks up a 1/d₁), and evolving the generator for
# unit time turns rates into transfer probabilities, with leakage and seepage competing.
Gamma = leakage_rate + seepage_rate
transfer = 1 - jnp.exp(-jnp.asarray(Gamma))
L1_expected = (leakage_rate / Gamma) * transfer / d1
L2_expected = (seepage_rate / Gamma) * transfer

rows = [("L₁ (leakage rate)", L1, L1_expected), ("L₂ (seepage rate)", L2, L2_expected)]
header = f"{'Parameter':<25} {'Extracted':>10} {'Expected':>10} {'Rel Err':>10}"
print(header + "\n" + "-" * len(header))
for name, ext, expected in rows:
    print(f"{name:<25} {100 * ext:>9.4f}% {100 * expected:>9.4f}% {abs(ext - expected) / expected:>9.1%}")
print(f"{'E (gate infidelity)':<25} {100 * E:>9.4f}%")
print(f"{'F̄ (gate fidelity)':<25} {100 * F_avg:>9.4f}%")

# --- Plot fits overlaid on data ---
m_fine = jnp.linspace(0, max(depths), 300)
plot_lrb(
    depths,
    p_comp,
    p0,
    fit_left=(m_fine, leakage_decay(popt_leak, m_fine), f"Fit: λ₁={lam1:.4f}"),
    fit_right=(m_fine, fidelity_decay(popt_fid, m_fine), f"Fit: λ₂={lam2:.4f}"),
)